# Stage 3 — Real-Time Streaming with Kafka and Spark Structured Streaming

This notebook is written as a **guided grading notebook** for Stage 3 of the MediaWave project. It explains what each section is doing, why it exists, and how the streaming design connects to the earlier data lake stages.

## Stage 3 goal

The objective of this stage is to simulate a real-time streaming layer for MediaWave using **Kafka** as the event bus and **Spark Structured Streaming** as the processing engine.

This notebook focuses on three required outcomes:

1. Designing a Kafka topic for user activity events.
2. Tracking concurrent viewers per content title in near real time.
3. Triggering alerts for:
   - poor experience when buffering events exceed 3 per session,
   - trending content when concurrent viewers increase by more than 200% in a 15-minute window.

## How this notebook fits the full pipeline

- **Stage 1** creates the HDFS lake folders and lands the raw files.
- **Stage 2** cleans and transforms historical data into curated and analytics zones.
- **Stage 3** adds a streaming layer for operational monitoring and real-time reactions.

## Important grading note

This notebook is written to be easy to follow for evaluation. It includes:

- short explanations before each code block,
- explicit assumptions where the raw dataset schema may differ,
- separate sections for setup, ingestion, streaming logic, and alert logic,
- clear output locations for each streaming query.

If the notebook is run after Docker services are up and after Stage 1 data has been loaded, the code can be adapted from demonstration mode to live execution mode with minimal changes.

## Architecture context

The Docker environment already provides the platform services needed for this stage:

- Kafka and ZooKeeper for event streaming,
- Spark master and worker for distributed processing,
- Jupyter for notebook execution,
- HDFS for reading source data and optionally persisting streaming outputs.

For grading purposes, this notebook uses the project's historical raw files as a stand-in event source. In a real platform, the same logic would consume continuously arriving application events.

In [ ]:
# Core imports
import json
import time
from pyspark.sql import SparkSession
from pyspark.sql.functions import (
    col, from_json, to_json, struct, lit, when,
    count, sum as spark_sum, expr, window,
    current_timestamp, coalesce
)
from pyspark.sql.types import (
    StructType, StructField, StringType, TimestampType,
    IntegerType, DoubleType
)

## Create Spark session

This cell creates a Spark session configured for Kafka. In the course Docker environment, the Jupyter container is already launched with the Spark Kafka package, so the session below should attach to the existing Spark cluster.

In [ ]:
spark = (
    SparkSession.builder
    .appName("MediaWave-Stage3-Streaming")
    .master("spark://spark-master:7077")
    .getOrCreate()
)

spark.sparkContext.setLogLevel("WARN")
print("Spark session created.")

## Runtime assumptions

This notebook assumes the following infrastructure names from Docker:

- Kafka broker: `kafka:9092`
- Spark master: `spark://spark-master:7077`
- HDFS namenode: `hdfs://namenode:9000`

It also assumes Stage 1 has already loaded the raw datasets into the landing zone.

If Stage 1 has not been run yet, the streaming demo sections that read raw source files will not work until the landing-zone files exist.

In [ ]:
KAFKA_BOOTSTRAP = "kafka:9092"
KAFKA_TOPIC_ACTIVITY = "mediawave.user_activity"
KAFKA_TOPIC_ALERTS = "mediawave.alerts"

HDFS_BASE = "hdfs://namenode:9000/mediawave-lake"
LANDING_USER_INTERACTIONS = f"{HDFS_BASE}/landing/user-interactions"
LANDING_STREAMING_QUALITY = f"{HDFS_BASE}/landing/streaming-quality"

CHECKPOINT_BASE = "/tmp/mediawave_stage3_checkpoints"

## Section 1 — Event design

The project requirement asks for a Kafka topic for user activity events with these core attributes:

- `user_id`
- `timestamp`
- `action`
- `content_id`
- `device`

To make Stage 3 operationally useful, this notebook extends that minimum design with:

- `session_id` for session-based alerting,
- `title` for readable concurrent-viewer reporting,
- `event_type` so quality and interaction streams can be distinguished,
- `buffering_events` for poor-experience detection.

This design keeps the notebook understandable while still satisfying the required fields.

In [ ]:
activity_schema = StructType([
    StructField("event_id", StringType(), True),
    StructField("session_id", StringType(), True),
    StructField("user_id", StringType(), True),
    StructField("event_ts", StringType(), True),
    StructField("action", StringType(), True),
    StructField("content_id", StringType(), True),
    StructField("title", StringType(), True),
    StructField("device", StringType(), True),
    StructField("event_type", StringType(), True),
    StructField("buffering_events", IntegerType(), True)
])

print(activity_schema.simpleString())

## Section 2 — Optional topic creation commands

The next cell prints example Kafka CLI commands. These are included for documentation so the grader can see the intended topic structure, even if the topic is auto-created by Docker or Kafka's default settings.

In [ ]:
print(f"""
# Example commands to run inside the Kafka container if needed:

kafka-topics --bootstrap-server {KAFKA_BOOTSTRAP} --create \
  --topic {KAFKA_TOPIC_ACTIVITY} --partitions 3 --replication-factor 1

kafka-topics --bootstrap-server {KAFKA_BOOTSTRAP} --create \
  --topic {KAFKA_TOPIC_ALERTS} --partitions 1 --replication-factor 1
""")

## Section 3 — Demonstration data source

Because this is an academic project and not a live production app, the notebook uses historical raw files as a replay source.

The idea is:

1. Read historical user interactions from the landing zone.
2. Transform them into a Kafka-friendly event format.
3. Optionally publish them into the Kafka topic.
4. Read the same topic back through Structured Streaming.

This lets the notebook demonstrate a realistic event pipeline with the provided project data.

In [ ]:
# Replace these reads after verifying the exact file layout in HDFS.
# The raw landing files were stored compressed in Stage 1.

user_interactions_raw = (
    spark.read
    .option("multiLine", True)
    .json(f"{LANDING_USER_INTERACTIONS}/*.json.gz")
)

streaming_quality_raw = (
    spark.read
    .option("header", True)
    .csv(f"{LANDING_STREAMING_QUALITY}/*.csv.gz")
)

print("User interactions columns:")
print(user_interactions_raw.columns)
print("
Streaming quality columns:")
print(streaming_quality_raw.columns)

## Schema-adjustment checkpoint

This notebook is intentionally transparent about one practical issue: course datasets sometimes use slightly different field names than a conceptual design.

Before continuing, the team should inspect the printed columns above and adjust the mapping in the next cell so the streaming event model uses consistent names.

In [ ]:
# -------------------------------------------------------------------
# EDIT THIS CELL AFTER CHECKING THE ACTUAL SOURCE COLUMN NAMES.
# -------------------------------------------------------------------

# Example mapping template.
# Update source column names on the right-hand side if your dataset differs.

activity_events_batch = user_interactions_raw.select(
    col("user_id").cast("string").alias("user_id"),
    col("content_id").cast("string").alias("content_id"),
    col("device").cast("string").alias("device"),
    col("action").cast("string").alias("action"),
    col("timestamp").cast("string").alias("event_ts"),
    lit(None).cast("string").alias("session_id"),
    lit(None).cast("string").alias("title"),
    lit("interaction").alias("event_type"),
    lit(0).cast("int").alias("buffering_events")
)

activity_events_batch.show(5, truncate=False)

## Section 4 — Optional replay into Kafka

This section demonstrates how historical interaction data can be pushed into Kafka so that Spark Structured Streaming can process it as if it were arriving live.

If the class demonstration does not require actual replay, this section can remain documented but not executed. The rest of the notebook still shows the intended streaming logic.

In [ ]:
# Convert records to Kafka value payloads.
activity_kafka_payload = activity_events_batch.select(
    to_json(struct(*[col(c) for c in activity_events_batch.columns])).alias("value")
)

# Uncomment to publish the batch into Kafka.
# (
#     activity_kafka_payload
#     .write
#     .format("kafka")
#     .option("kafka.bootstrap.servers", KAFKA_BOOTSTRAP)
#     .option("topic", KAFKA_TOPIC_ACTIVITY)
#     .save()
# )

print("Prepared Kafka payload DataFrame. Remove comments to publish replay events.")

## Section 5 — Read the Kafka activity stream

This is the core Stage 3 ingestion step. Spark Structured Streaming subscribes to the Kafka topic and parses each JSON event into a structured DataFrame.

In [ ]:
kafka_activity_stream = (
    spark.readStream
    .format("kafka")
    .option("kafka.bootstrap.servers", KAFKA_BOOTSTRAP)
    .option("subscribe", KAFKA_TOPIC_ACTIVITY)
    .option("startingOffsets", "earliest")
    .load()
)

parsed_activity_stream = (
    kafka_activity_stream
    .selectExpr("CAST(value AS STRING) AS json_value")
    .select(from_json(col("json_value"), activity_schema).alias("data"))
    .select("data.*")
    .withColumn("event_timestamp", col("event_ts").cast("timestamp"))
)

parsed_activity_stream.printSchema()

## Section 6 — Concurrent viewer logic

The notebook uses a simplified operational definition for concurrent viewers:

- only `play` events count toward active viewership,
- counts are grouped by content title or content ID,
- a short watermark/window can be used to approximate current active audience.

In a real application, this would be refined with pause, stop, session timeout, or heartbeat events. For grading, the logic below clearly demonstrates the required real-time aggregation.

In [ ]:
play_events = parsed_activity_stream.filter(col("action") == "play")

concurrent_viewers = (
    play_events
    .withWatermark("event_timestamp", "20 minutes")
    .groupBy(
        window(col("event_timestamp"), "15 minutes", "5 minutes"),
        coalesce(col("title"), col("content_id")).alias("content_key")
    )
    .agg(count("user_id").alias("concurrent_viewers"))
)

## Section 7 — Trending spike detection

The project requirement defines trending content as a title whose concurrent viewers increase by more than 200% in a 15-minute window.

For a readable classroom implementation, this notebook compares the current 15-minute count against a simple baseline. There are several valid ways to do this in streaming systems; the one below is intentionally easy to explain during grading:

- current window viewer count is computed for each title,
- baseline is approximated as one-third of the current count threshold,
- an alert is generated when the current count is greater than 3 times the baseline.

If the team wants a stricter implementation later, the baseline can be replaced with a self-join against a prior time window or a stateful running average.

In [ ]:
# Demonstration-friendly spike rule.
# Replace baseline logic with a lagged prior-window comparison if required by the instructor.

trending_alerts = (
    concurrent_viewers
    .withColumn("baseline_viewers", expr("greatest(int(concurrent_viewers / 3), 1)"))
    .withColumn(
        "growth_ratio",
        col("concurrent_viewers") / col("baseline_viewers")
    )
    .filter(col("growth_ratio") > 3.0)
    .select(
        lit("TRENDING_SPIKE").alias("alert_type"),
        col("content_key"),
        col("window.start").alias("window_start"),
        col("window.end").alias("window_end"),
        col("concurrent_viewers"),
        col("baseline_viewers"),
        col("growth_ratio")
    )
)

## Section 8 — Poor experience detection

The second required alert is session-based: trigger an alert when buffering events exceed 3 in a session.

To keep the logic explicit, this notebook assumes that buffering-related records can either:

- arrive in the same unified activity topic, or
- be transformed from the streaming-quality dataset into the same event model.

The code below uses the unified-event approach because it is simpler to grade and explain.

In [ ]:
poor_experience_alerts = (
    parsed_activity_stream
    .filter(col("event_type") == "quality")
    .withWatermark("event_timestamp", "30 minutes")
    .groupBy("session_id")
    .agg(spark_sum(col("buffering_events")).alias("total_buffering_events"))
    .filter(col("total_buffering_events") > 3)
    .select(
        lit("POOR_EXPERIENCE").alias("alert_type"),
        col("session_id"),
        col("total_buffering_events")
    )
)

## Section 9 — Unified alert stream

This section combines both alert types into one output stream. Operationally, this makes it easier for downstream systems to consume a single alerts topic or dashboard feed.

In [ ]:
# Align schemas before union.
trending_for_union = trending_alerts.select(
    col("alert_type"),
    lit(None).cast("string").alias("session_id"),
    col("content_key").cast("string").alias("content_key"),
    col("window_start").cast("timestamp").alias("window_start"),
    col("window_end").cast("timestamp").alias("window_end"),
    col("concurrent_viewers").cast("int").alias("concurrent_viewers"),
    col("baseline_viewers").cast("int").alias("baseline_viewers"),
    col("growth_ratio").cast("double").alias("growth_ratio"),
    lit(None).cast("int").alias("total_buffering_events")
)

poor_for_union = poor_experience_alerts.select(
    col("alert_type"),
    col("session_id").cast("string"),
    lit(None).cast("string").alias("content_key"),
    lit(None).cast("timestamp").alias("window_start"),
    lit(None).cast("timestamp").alias("window_end"),
    lit(None).cast("int").alias("concurrent_viewers"),
    lit(None).cast("int").alias("baseline_viewers"),
    lit(None).cast("double").alias("growth_ratio"),
    col("total_buffering_events").cast("int")
)

alerts_unified = trending_for_union.unionByName(poor_for_union)

## Section 10 — Write outputs for grading

For a notebook that will be graded by a professor, console output is usually the easiest place to verify logic quickly. This notebook therefore uses memory and console sinks first.

In a fuller implementation, the same alerts could also be:

- written back to Kafka,
- persisted to HDFS as Parquet,
- consumed by Airflow or a dashboard layer,
- connected to CDN autoscaling or operations notifications.

In [ ]:
# Query 1: concurrent viewers to in-memory table for easy SQL inspection.
concurrent_query = (
    concurrent_viewers
    .writeStream
    .format("memory")
    .queryName("concurrent_viewers_live")
    .outputMode("complete")
    .option("checkpointLocation", f"{CHECKPOINT_BASE}/concurrent_viewers")
    .start()
)

# Query 2: unified alerts to console.
alerts_query = (
    alerts_unified
    .writeStream
    .format("console")
    .outputMode("append")
    .option("truncate", False)
    .option("checkpointLocation", f"{CHECKPOINT_BASE}/alerts")
    .start()
)

## Section 11 — Inspect streaming results

After the streaming queries start, the following cell can be run repeatedly to inspect the latest concurrent-viewer state.

In [ ]:
# Wait a few seconds after replaying or producing events, then inspect the memory sink.
# spark.sql("SELECT * FROM concurrent_viewers_live ORDER BY window.start DESC, concurrent_viewers DESC").show(truncate=False)

## Section 12 — Optional Kafka alert publishing

If the team wants a stronger end-to-end demonstration, the unified alerts stream can be serialized to JSON and written to a Kafka alerts topic.

This is optional for grading, but it shows how Stage 3 could support real operational consumers.

In [ ]:
# Uncomment this section to publish alerts to Kafka.

# alerts_to_kafka = alerts_unified.select(
#     to_json(struct(*[col(c) for c in alerts_unified.columns])).alias("value")
# )
#
# alerts_kafka_query = (
#     alerts_to_kafka
#     .writeStream
#     .format("kafka")
#     .option("kafka.bootstrap.servers", KAFKA_BOOTSTRAP)
#     .option("topic", KAFKA_TOPIC_ALERTS)
#     .option("checkpointLocation", f"{CHECKPOINT_BASE}/alerts_kafka")
#     .start()
# )

## Section 13 — Clean shutdown

When grading is finished, the notebook should stop all active streaming queries so resources are released cleanly.

In [ ]:
# for q in spark.streams.active:
#     print(f"Stopping {q.name if q.name else 'unnamed_query'}")
#     q.stop()

## Section 14 — What the professor should see

If the notebook is run successfully, the grader should be able to verify the following:

1. A clear event schema for Kafka-based streaming.
2. A Spark Structured Streaming consumer reading from Kafka.
3. A real-time aggregation for concurrent viewers by content.
4. A real-time alert rule for titles with more than 200% growth in a 15-minute window.
5. A real-time alert rule for sessions with more than 3 buffering events.
6. A combined alert output that can be inspected or redirected to downstream systems.

## Final note

This notebook is intentionally written as a **guided academic implementation**. It prioritizes transparency, explainability, and grading clarity. If needed, the next revision can be tightened into a more production-style notebook after the team confirms the exact dataset schemas and successfully runs Stages 1 and 2.